# Agent 工作流程

深入 Agent 内部，理解模型和工具之间如何协作、Agent 什么时候停止。

In [ ]:
from dotenv import load_dotenv
load_dotenv()

print("环境变量已加载")

## Agent 执行循环

Agent 的核心是一个循环：**调用模型 → 检查是否需要工具 → 执行工具 → 重复**，直到模型不再请求工具调用。

In [ ]:
from langchain.tools import tool
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage

@tool
def get_weather(city: str) -> str:
    """查询指定城市的天气。"""
    data = {"杭州": "晴，25°C", "北京": "多云，18°C"}
    return data.get(city, f"未找到 {city}")

@tool
def get_time(city: str) -> str:
    """查询指定城市的当前时间。"""
    data = {"杭州": "14:30", "北京": "14:30", "纽约": "02:30"}
    return data.get(city, f"未找到 {city}")

model = init_chat_model("deepseek:deepseek-v4-flash", temperature=0)
agent = create_agent(
    model=model, tools=[get_weather, get_time],
    system_prompt="你是一个乐于助人的助手。",
)

# 使用 stream_mode="updates" 追踪每一步
print("=== 使用 stream() 追踪执行过程 ===\n")
for chunk in agent.stream(
    {"messages": [HumanMessage(content="杭州现在天气怎么样？几点了？")]},
    stream_mode="updates",
):
    for node_name, update in chunk.items():
        print(f"--- 节点: {node_name} ---")
        if "messages" in update:
            for msg in update["messages"]:
                if hasattr(msg, 'tool_calls') and msg.tool_calls:
                    for tc in msg.tool_calls:
                        print(f"  → 请求调用: {tc['name']}({tc['args']})")
                elif msg.type == "tool":
                    print(f"  → 工具结果 [{msg.name}]: {msg.content}")
                elif msg.type == "ai" and msg.content:
                    print(f"  → 回复: {msg.content[:100]}")

## stream_mode 详解

| 模式 | 返回内容 | 适用场景 |
| --- | --- | --- |
| updates | 每个节点执行后的状态更新 | 追踪 Agent 执行步骤 |
| values | 每个节点后的完整状态 | 查看每一步的完整消息历史 |
| messages | 逐 Token 的消息流 | 前端流式展示打字效果 |
| custom | 自定义事件 | Middleware 发送自定义事件 |

In [ ]:
# stream_mode="values"——查看完整状态变化
print("=== stream_mode='values' ===\n")
for i, chunk in enumerate(agent.stream(
    {"messages": [HumanMessage(content="杭州天气怎么样？")]},
    stream_mode="values",
)):
    messages = chunk.get("messages", [])
    print(f"状态 {i}: {len(messages)} 条消息")
    for msg in messages:
        print(f"  [{msg.type}] {str(msg.content)[:80]}")
    if i >= 3:
        break

In [ ]:
# stream_mode="messages"——逐 Token 打字效果
print("=== stream_mode='messages' ===\n")
for msg_chunk, metadata in agent.stream(
    {"messages": [HumanMessage(content="用一句话介绍菜鸟教程 RUNOOB")]},
    stream_mode="messages",
):
    if hasattr(msg_chunk, 'content') and msg_chunk.content:
        print(msg_chunk.content, end="", flush=True)
print()

## Agent 的退出条件

| 条件 | 说明 |
| --- | --- |
| 无工具调用 | 模型认为任务完成，直接回复 |
| return_direct=True | 工具执行后立即结束 |
| structured_response | 结构化输出完成 |
| jump_to="end" | Middleware 主动结束 |

In [ ]:
# 对比：无工具 vs 有工具
model = init_chat_model("deepseek:deepseek-v4-flash", temperature=0)

# 无工具：模型直接回复，循环只执行一次
agent_no_tools = create_agent(model=model)
result = agent_no_tools.invoke({
    "messages": [HumanMessage(content="用一句话介绍菜鸟教程")]
})
print(f"无工具场景，消息数: {len(result['messages'])}")  # 2：human + ai

# 有工具：多轮循环
@tool
def search_course(keyword: str) -> str:
    """搜索菜鸟教程课程"""
    return f"找到 {keyword} 相关课程 3 门"

agent_with_tools = create_agent(model=model, tools=[search_course])
result = agent_with_tools.invoke({
    "messages": [HumanMessage(content="搜索 Python 课程")]
})
print(f"有工具场景，消息数: {len(result['messages'])}")  # 4：human+ai(tool_call)+tool+ai
for msg in result["messages"]:
    print(f"  [{msg.type}]", end="")
    if hasattr(msg, 'tool_calls') and msg.tool_calls:
        print(f" tool_calls: {[tc['name'] for tc in msg.tool_calls]}")
    else:
        print(f" {str(msg.content)[:60]}")

## config——线程 ID 与运行时配置

使用 checkpointer 时，通过 `thread_id` 区分不同对话线程。

**术语：**

- **stream_mode**：流式模式，控制返回数据的粒度
- **updates**：节点级状态更新模式
- **values**：完整状态值模式
- **messages**：逐 Token 消息流模式
- **Agent 执行循环**：模型调用 → 工具执行 → 循环直到完成